In [2]:
import os
import json
import numpy as np
path="/kaggle/input/datasets/hasyimabdillah/workoutfitness-video"

In [3]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.0 MB/s eta 0:00:0000:01


In [3]:
from ultralytics import YOLO

model = YOLO("yolo11n-pose.pt")
CONF_THRESHOLD=0.3

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
import cv2



def compute_elbow_body_ratio(video_path):
    """
    ratio = dist(elbow, hip) / dist(shoulder, hip)
    
    Uses shoulder-to-hip as normalization instead of nose-to-ankle
    because ankle and nose are often out of frame in close-up videos.
    
    Picks the side with higher elbow confidence per frame.
    """
    results = model.track(source=video_path, stream=True, verbose=False)
    ratios = []

    for frame_result in results:
        if frame_result.keypoints is None:
            ratios.append(None)
            continue
        if len(frame_result.keypoints.xy) == 0:
            ratios.append(None)
            continue

        xy   = frame_result.keypoints.xy[0].cpu().numpy()
        conf = frame_result.keypoints.conf[0].cpu().numpy()

        # Pick side with higher elbow confidence
        use_left     = conf[7] >= conf[8]
        elbow_idx    = 7  if use_left else 8
        shoulder_idx = 5  if use_left else 6
        hip_idx      = 11 if use_left else 12

        needed = [elbow_idx, shoulder_idx, hip_idx]
        if any(conf[i] < CONF_THRESHOLD for i in needed):
            ratios.append(None)
            continue

        elbow    = xy[elbow_idx]
        shoulder = xy[shoulder_idx]
        hip      = xy[hip_idx]

        torso_height = np.linalg.norm(shoulder - hip) + 1e-6
        elbow_dist   = np.linalg.norm(elbow - hip)
        ratio        = round(float(elbow_dist / torso_height), 4)

        ratios.append(ratio)

    return ratios

In [5]:
def get_videos(folder, n=5):
    p = os.path.join(path, folder)
    videos = sorted([
        os.path.join(p, f)
        for f in os.listdir(p)
        if f.endswith(".mp4")
    ])
    return videos[:n]


In [14]:
def compute_alignment_deviation(video_path, mode):
    """
    Generic function for body alignment deviation.

    mode = "trunk_lean":
        Measures horizontal lean of trunk
        Uses: shoulder, hip
        For: bicep curl, hammer curl, tricep pushdown

    mode = "hip_alignment":
        Measures vertical deviation of hip from
        shoulder-ankle line
        Uses: shoulder, hip, ankle
        For: pushup, plank

    Returns list of deviation values per frame:
        trunk_lean:
            positive = leaning forward
            negative = leaning back
            0 = perfectly upright

        hip_alignment:
            positive = hip sagging below line
            negative = hip piking above line
            0 = perfect plank position
    """
    results = model.track(source=video_path, stream=True, verbose=False)
    deviations = []

    for frame_result in results:
        if frame_result.keypoints is None:
            deviations.append(None)
            continue
        if len(frame_result.keypoints.xy) == 0:
            deviations.append(None)
            continue

        xy   = frame_result.keypoints.xy[0].cpu().numpy()
        conf = frame_result.keypoints.conf[0].cpu().numpy()

        use_left     = conf[7] >= conf[8]
        shoulder_idx = 5  if use_left else 6
        hip_idx      = 11 if use_left else 12
        ankle_idx    = 15 if use_left else 16

        if mode == "trunk_lean":
            needed = [shoulder_idx, hip_idx]
            if any(conf[i] < CONF_THRESHOLD for i in needed):
                deviations.append(None)
                continue

            shoulder     = xy[shoulder_idx]
            hip          = xy[hip_idx]
            torso_length = np.linalg.norm(shoulder - hip) + 1e-6

            # horizontal displacement normalized by torso length
            # positive = shoulder in front of hip = leaning forward
            # negative = shoulder behind hip = leaning back
            deviation = round(
                float((shoulder[0] - hip[0]) / torso_length), 4
            )

        elif mode == "hip_alignment":
            needed = [shoulder_idx, hip_idx, ankle_idx]
            if any(conf[i] < CONF_THRESHOLD for i in needed):
                deviations.append(None)
                continue

            shoulder = xy[shoulder_idx]
            hip      = xy[hip_idx]
            ankle    = xy[ankle_idx]

            # Expected hip_y = point on line between shoulder and ankle
            # interpolated at hip_x position
            body_length = np.linalg.norm(shoulder - ankle) + 1e-6

            # t = how far along shoulder→ankle is the hip x position
            t = (hip[0] - shoulder[0]) / (ankle[0] - shoulder[0] + 1e-6)
            expected_hip_y = shoulder[1] + t * (ankle[1] - shoulder[1])

            # deviation normalized by body length
            # positive = hip below line = sagging
            # negative = hip above line = piking
            deviation = round(
                float((hip[1] - expected_hip_y) / body_length), 4
            )

        deviations.append(deviation)

    return deviations



In [ ]:
import json

hammer_videos  = get_videos("hammer curl")
barbell_videos = get_videos("barbell biceps curl")
tricep_videos  = get_videos("tricep Pushdown")

# ── Collect all ratios ────────────────────────────────────────
all_ratios = []

all_videos = [
    ("hammer_curl",        hammer_videos),
    ("barbell_biceps_curl", barbell_videos),
    ("tricep_pushdown",    tricep_videos),
]

for exercise_name, video_list in all_videos:
    print(f"\nProcessing {exercise_name}...")
    for path in video_list:
        ratios = compute_elbow_body_ratio(path)
        valid  = [r for r in ratios if r is not None]
        print(f"  {path.split('/')[-1]}: {len(valid)} valid frames")
        all_ratios.extend(valid)

print(f"\nTotal valid frames: {len(all_ratios)}")

# ── Compute threshold ─────────────────────────────────────────
threshold = round(float(np.percentile(all_ratios, 90)), 4)

print(f"90th percentile threshold: {threshold}")

# ── Save reference ────────────────────────────────────────────
reference = {
    "curl_pushdown": threshold
}

with open("curl_pushdown_reference.json", "w") as f:
    json.dump(reference, f, indent=2)

print(json.dumps(reference, indent=2))



In [ ]:
all_leans = []

for exercise_name, video_list in all_videos:
    print(f"\nProcessing {exercise_name}...")
    for p in video_list:
        leans = compute_alignment_deviation(p, "trunk_lean")
        valid = [l for l in leans if l is not None]
        print(f"  {os.path.basename(p)}: {len(valid)} valid frames")
        all_leans.extend(valid)

print(f"\nTotal valid frames: {len(all_leans)}")

all_leans_arr = np.array(all_leans)
print(f"min:             {round(float(np.min(all_leans_arr)), 4)}")
print(f"25th percentile: {round(float(np.percentile(all_leans_arr, 25)), 4)}")
print(f"50th percentile: {round(float(np.percentile(all_leans_arr, 50)), 4)}")
print(f"75th percentile: {round(float(np.percentile(all_leans_arr, 75)), 4)}")
print(f"90th percentile: {round(float(np.percentile(all_leans_arr, 90)), 4)}")
print(f"95th percentile: {round(float(np.percentile(all_leans_arr, 95)), 4)}")
print(f"max:             {round(float(np.max(all_leans_arr)), 4)}")

# Load existing reference and add trunk lean
with open("curl_pushdown_reference.json") as f:
    reference = json.load(f)

reference["trunk_lean_forward_max"] = round(float(np.percentile(all_leans_arr, 75)), 4)
reference["trunk_lean_back_min"]    = round(float(np.percentile(all_leans_arr, 25)), 4)

with open("curl_pushdown_reference.json", "w") as f:
    json.dump(reference, f, indent=2)

print(json.dumps(reference, indent=2))


In [ ]:
import cv2
import numpy as np
import json
from PIL import Image
import os

# ── Load reference ────────────────────────────────────────────
with open("curl_pushdown_reference.json") as f:
    reference = json.load(f)

ELBOW_THRESHOLD  = reference["curl_pushdown"]          + 0.05
LEAN_FORWARD_MAX = reference["trunk_lean_forward_max"] + 0.05
LEAN_BACK_MIN    = reference["trunk_lean_back_min"]    - 0.05
CONF_THRESHOLD      = 0.3
DEBOUNCE_FRAMES     = 4

# ── Load GIF frames ───────────────────────────────────────────
gif_path ="/kaggle/input/datasets/abdallahgalalll/test-set/bicep_curl.gif"

gif = Image.open(gif_path)
frames = []
try:
    while True:
        frame_rgb = gif.convert("RGB")
        frame_np  = np.array(frame_rgb)
        frame_bgr = cv2.cvtColor(frame_np, cv2.COLOR_RGB2BGR)
        frames.append(frame_bgr)
        gif.seek(gif.tell() + 1)
except EOFError:
    pass

print(f"GIF has {len(frames)} frames")

# ── Debounce counters — one per violation type ─────────────────
elbow_count        = 0
lean_forward_count = 0
lean_back_count    = 0

elbow_active        = False
lean_forward_active = False
lean_back_active    = False

output_frames = []

for i, frame in enumerate(frames):
    result = model(frame, verbose=False)

    elbow_ratio  = None
    trunk_lean   = None
    elbow_pt     = None
    hip_pt       = None
    shoulder_pt  = None
    kp_xy        = None
    kp_conf      = None

    if result and result[0].keypoints is not None:
        if len(result[0].keypoints.xy) > 0:
            xy   = result[0].keypoints.xy[0].cpu().numpy()
            conf = result[0].keypoints.conf[0].cpu().numpy()
            kp_xy   = xy
            kp_conf = conf

            use_left     = conf[7] >= conf[8]
            elbow_idx    = 7  if use_left else 8
            shoulder_idx = 5  if use_left else 6
            hip_idx      = 11 if use_left else 12

            needed = [elbow_idx, shoulder_idx, hip_idx]
            if all(conf[i] >= CONF_THRESHOLD for i in needed):
                elbow    = xy[elbow_idx]
                shoulder = xy[shoulder_idx]
                hip      = xy[hip_idx]

                torso = np.linalg.norm(shoulder - hip) + 1e-6

                # Elbow ratio
                elbow_ratio = round(float(np.linalg.norm(elbow - hip) / torso), 4)

                # Trunk lean
                trunk_lean = round(float((shoulder[0] - hip[0]) / torso), 4)

                elbow_pt    = (int(elbow[0]),    int(elbow[1]))
                hip_pt      = (int(hip[0]),       int(hip[1]))
                shoulder_pt = (int(shoulder[0]),  int(shoulder[1]))

    # ── Debounce — elbow flare ────────────────────────────────
    if elbow_ratio is not None and elbow_ratio > ELBOW_THRESHOLD:
        elbow_count += 1
    else:
        elbow_count  = 0
        elbow_active = False
    if elbow_count >= DEBOUNCE_FRAMES:
        elbow_active = True

    # ── Debounce — lean forward ───────────────────────────────
    if trunk_lean is not None and trunk_lean > LEAN_FORWARD_MAX:
        lean_forward_count += 1
    else:
        lean_forward_count  = 0
        lean_forward_active = False
    if lean_forward_count >= DEBOUNCE_FRAMES:
        lean_forward_active = True

    # ── Debounce — lean back ──────────────────────────────────
    if trunk_lean is not None and trunk_lean < LEAN_BACK_MIN:
        lean_back_count += 1
    else:
        lean_back_count  = 0
        lean_back_active = False
    if lean_back_count >= DEBOUNCE_FRAMES:
        lean_back_active = True

    # ── Draw on frame ─────────────────────────────────────────
    annotated = frame.copy()

    # Draw keypoints
    if kp_xy is not None:
        for idx in [5, 6, 7, 8, 11, 12]:
            if kp_conf[idx] >= CONF_THRESHOLD:
                pt = (int(kp_xy[idx][0]), int(kp_xy[idx][1]))
                cv2.circle(annotated, pt, 5, (0, 255, 0), -1)

    # Draw elbow-hip line
    if elbow_pt and hip_pt:
        color = (0, 0, 255) if elbow_active else (0, 255, 0)
        cv2.line(annotated, elbow_pt, hip_pt, color, 2)
        cv2.circle(annotated, elbow_pt, 8, color, -1)

    # Draw shoulder-hip line (trunk)
    if shoulder_pt and hip_pt:
        color = (0, 0, 255) if (lean_forward_active or lean_back_active) else (255, 165, 0)
        cv2.line(annotated, shoulder_pt, hip_pt, color, 2)

    # ── Ratio values display ──────────────────────────────────
    if elbow_ratio is not None:
        cv2.putText(annotated,
            f"elbow ratio: {elbow_ratio:.3f} / {ELBOW_THRESHOLD}",
            (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 2)
    if trunk_lean is not None:
        cv2.putText(annotated,
            f"lean: {trunk_lean:.3f} [{LEAN_BACK_MIN},{LEAN_FORWARD_MAX}]",
            (10, 48), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 2)

    # ── Feedback banners ──────────────────────────────────────
    banner_y = 55
    any_violation = elbow_active or lean_forward_active or lean_back_active

    if elbow_active:
        cv2.rectangle(annotated, (0, banner_y), (annotated.shape[1], banner_y+35), (0,0,200), -1)
        cv2.putText(annotated, "WARNING: Keep elbows close to body!",
            (10, banner_y+24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
        banner_y += 38

    if lean_forward_active:
        cv2.rectangle(annotated, (0, banner_y), (annotated.shape[1], banner_y+35), (0,100,200), -1)
        cv2.putText(annotated, "WARNING: Do not lean forward!",
            (10, banner_y+24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
        banner_y += 38

    if lean_back_active:
        cv2.rectangle(annotated, (0, banner_y), (annotated.shape[1], banner_y+35), (0,100,200), -1)
        cv2.putText(annotated, "WARNING: Do not lean back!",
            (10, banner_y+24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
        banner_y += 38

    if not any_violation:
        cv2.rectangle(annotated, (0, banner_y), (annotated.shape[1], banner_y+35), (0,150,0), -1)
        cv2.putText(annotated, "Good form",
            (10, banner_y+24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

    output_frames.append(annotated)

# ── Save output ───────────────────────────────────────────────
h, w = output_frames[0].shape[:2]
out  = cv2.VideoWriter(
    "feedback_output_v2.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    10,
    (w, h)
)
for frame in output_frames:
    out.write(frame)
out.release()
print("Saved: feedback_output_v2.mp4")


In [ ]:
all_deviations = []

all_videos_plank_pushup = [
    ("push_up", pushup_videos),
    ("plank",   plank_videos),
]

for exercise_name, video_list in all_videos_plank_pushup:
    print(f"\nProcessing {exercise_name}...")
    for p in video_list:
        devs  = compute_alignment_deviation(p, "hip_alignment")
        valid = [d for d in devs if d is not None]
        print(f"  {os.path.basename(p)}: {len(valid)} valid frames")
        all_deviations.extend(valid)

print(f"\nTotal valid frames: {len(all_deviations)}")

all_devs_arr = np.array(all_deviations)
print(f"min:             {round(float(np.min(all_devs_arr)), 4)}")
print(f"25th percentile: {round(float(np.percentile(all_devs_arr, 25)), 4)}")
print(f"50th percentile: {round(float(np.percentile(all_devs_arr, 50)), 4)}")
print(f"75th percentile: {round(float(np.percentile(all_devs_arr, 75)), 4)}")
print(f"90th percentile: {round(float(np.percentile(all_devs_arr, 90)), 4)}")
print(f"95th percentile: {round(float(np.percentile(all_devs_arr, 95)), 4)}")
print(f"max:             {round(float(np.max(all_devs_arr)), 4)}")

In [ ]:
plank_pushup_reference = {
    "hip_sag_max":  round(float(np.percentile(all_devs_arr, 75)), 4),
    "hip_pike_min": round(float(np.percentile(all_devs_arr, 25)), 4)
}

with open("plank_pushup_reference.json", "w") as f:
    json.dump(plank_pushup_reference, f, indent=2)

print(json.dumps(plank_pushup_reference, indent=2))

In [ ]:
import cv2
import numpy as np
import json
from PIL import Image
import os

# ── Load reference ────────────────────────────────────────────
with open("plank_pushup_reference.json") as f:
    reference = json.load(f)

HIP_SAG_MAX  = reference["hip_sag_max"]  + 0.05
HIP_PIKE_MIN = reference["hip_pike_min"]
print(HIP_PIKE_MIN)
CONF_THRESHOLD  = 0.5
DEBOUNCE_FRAMES = 2

# ── Load GIF frames ───────────────────────────────────────────
gif_path = "/kaggle/input/datasets/abdallahgalalll/test-set/plank.gif"

gif = Image.open(gif_path)
frames = []
try:
    while True:
        frame_rgb = gif.convert("RGB")
        frame_np  = np.array(frame_rgb)
        frame_bgr = cv2.cvtColor(frame_np, cv2.COLOR_RGB2BGR)
        frames.append(frame_bgr)
        gif.seek(gif.tell() + 1)
except EOFError:
    pass

print(f"GIF has {len(frames)} frames")

# ── Debounce counters ─────────────────────────────────────────
sag_count  = 0
pike_count = 0

sag_active  = False
pike_active = False

output_frames = []

for i, frame in enumerate(frames):
    result = model(frame, verbose=False)

    deviation    = None
    shoulder_pt  = None
    hip_pt       = None
    ankle_pt     = None
    kp_xy        = None
    kp_conf      = None

    if result and result[0].keypoints is not None:
        if len(result[0].keypoints.xy) > 0:
            xy   = result[0].keypoints.xy[0].cpu().numpy()
            conf = result[0].keypoints.conf[0].cpu().numpy()
            kp_xy   = xy
            kp_conf = conf

            use_left     = conf[7] >= conf[8]
            shoulder_idx = 5  if use_left else 6
            hip_idx      = 11 if use_left else 12
            ankle_idx    = 15 if use_left else 16

            needed = [shoulder_idx, hip_idx, ankle_idx]
            if all(conf[i] >= CONF_THRESHOLD for i in needed):
                shoulder = xy[shoulder_idx]
                hip      = xy[hip_idx]
                ankle    = xy[ankle_idx]

                horizontal_span = abs(ankle[0] - shoulder[0])
                body_length     = np.linalg.norm(shoulder - ankle) + 1e-6

                if horizontal_span >= 0.1 * body_length:
                    t = (hip[0] - shoulder[0]) / (ankle[0] - shoulder[0] + 1e-6)
                    if 0.0 <= t <= 1.0:
                        expected_hip_y = shoulder[1] + t * (ankle[1] - shoulder[1])
                        deviation = round(float((hip[1] - expected_hip_y) / body_length), 4)

                shoulder_pt = (int(shoulder[0]), int(shoulder[1]))
                hip_pt      = (int(hip[0]),      int(hip[1]))
                ankle_pt    = (int(ankle[0]),     int(ankle[1]))

    # ── Debounce — sag ────────────────────────────────────────
    if deviation is not None and deviation > HIP_SAG_MAX:
        sag_count += 1
    else:
        sag_count  = 0
        sag_active = False
    if sag_count >= DEBOUNCE_FRAMES:
        sag_active = True

    # ── Debounce — pike ───────────────────────────────────────
    if deviation is not None and deviation < HIP_PIKE_MIN:
        pike_count += 1
    else:
        pike_count  = 0
        pike_active = False
    if pike_count >= DEBOUNCE_FRAMES:
        pike_active = True

    # ── Draw on frame ─────────────────────────────────────────
    annotated = frame.copy()

    # Draw keypoints
    if kp_xy is not None:
        for idx in [5, 6, 7, 8, 11, 12, 15, 16]:
            if kp_conf[idx] >= CONF_THRESHOLD:
                pt = (int(kp_xy[idx][0]), int(kp_xy[idx][1]))
                cv2.circle(annotated, pt, 5, (0, 255, 0), -1)

    # Draw shoulder-hip-ankle line
    if shoulder_pt and hip_pt and ankle_pt:
        body_color = (0, 0, 255) if (sag_active or pike_active) else (0, 255, 0)
        cv2.line(annotated, shoulder_pt, ankle_pt, (255, 165, 0), 1)  # reference line
        cv2.line(annotated, shoulder_pt, hip_pt,   body_color,    2)  # actual trunk
        cv2.line(annotated, hip_pt,      ankle_pt, body_color,    2)  # actual lower
        cv2.circle(annotated, hip_pt, 8, body_color, -1)

    # ── Values display ────────────────────────────────────────
    if deviation is not None:
        cv2.putText(annotated,
            f"hip deviation: {deviation:.3f} [{HIP_PIKE_MIN:.3f}, {HIP_SAG_MAX:.3f}]",
            (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

    # ── Feedback banners ──────────────────────────────────────
    banner_y      = 35
    any_violation = sag_active or pike_active

    if sag_active:
        cv2.rectangle(annotated, (0, banner_y), (annotated.shape[1], banner_y+35), (0, 0, 200), -1)
        cv2.putText(annotated, "WARNING: Hips sagging — engage your core!",
            (10, banner_y+24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        banner_y += 38

    if pike_active:
        cv2.rectangle(annotated, (0, banner_y), (annotated.shape[1], banner_y+35), (0, 100, 200), -1)
        cv2.putText(annotated, "WARNING: Hips too high — lower your hips!",
            (10, banner_y+24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        banner_y += 38

    if not any_violation:
        cv2.rectangle(annotated, (0, banner_y), (annotated.shape[1], banner_y+35), (0, 150, 0), -1)
        cv2.putText(annotated, "Good form",
            (10, banner_y+24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    output_frames.append(annotated)

# ── Save output ───────────────────────────────────────────────
h, w = output_frames[0].shape[:2]
out  = cv2.VideoWriter(
    "feedback_pushup_plank.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    10,
    (w, h)
)
for frame in output_frames:
    out.write(frame)
out.release()
print("Saved: feedback_pushup_plank.mp4")

In [7]:
def compute_knee_width_ratio(video_path):
    """
    Measures knee width relative to hip width.
    
    ratio = dist(left_knee, right_knee) / dist(left_hip, right_hip)
    
    ratio < 1.0 → knees caving in  → ACL/MCL injury risk
    ratio ≈ 1.0 → knees aligned    → correct
    ratio > 1.5 → knees too wide   → hip impingement risk
    
    Requires FRONT VIEW — both knees must be visible.
    
    Keypoints:
      left_knee  = 13
      right_knee = 14
      left_hip   = 11
      right_hip  = 12
    """
    results = model.track(source=video_path, stream=True, verbose=False)
    ratios = []

    for frame_result in results:
        if frame_result.keypoints is None:
            ratios.append(None)
            continue
        if len(frame_result.keypoints.xy) == 0:
            ratios.append(None)
            continue

        xy   = frame_result.keypoints.xy[0].cpu().numpy()
        conf = frame_result.keypoints.conf[0].cpu().numpy()

        needed = [11, 12, 13, 14]
        if any(conf[i] < CONF_THRESHOLD for i in needed):
            ratios.append(None)
            continue

        left_knee  = xy[13]
        right_knee = xy[14]
        left_hip   = xy[11]
        right_hip  = xy[12]

        hip_width  = np.linalg.norm(left_hip - right_hip) + 1e-6
        knee_width = np.linalg.norm(left_knee - right_knee)
        ratio      = round(float(knee_width / hip_width), 4)

        ratios.append(ratio)

    return ratios

In [15]:
all_exercises = [
    ("squat",             squat_videos),
    ("deadlift",          deadlift_videos),
    ("romanian_deadlift", rdl_videos),
]

for exercise_name, video_list in all_exercises:
    ratios_all = []
    print(f"\n{exercise_name}:")
    for p in video_list:
        ratios = compute_knee_width_ratio(p)
        valid  = [r for r in ratios if r is not None]
        ratios_all.extend(valid)

    arr = np.array(ratios_all)
    print(f"  min:             {round(float(np.min(arr)), 4)}")
    print(f"  25th percentile: {round(float(np.percentile(arr, 25)), 4)}")
    print(f"  50th percentile: {round(float(np.percentile(arr, 50)), 4)}")
    print(f"  75th percentile: {round(float(np.percentile(arr, 75)), 4)}")
    print(f"  90th percentile: {round(float(np.percentile(arr, 90)), 4)}")
    print(f"  max:             {round(float(np.max(arr)), 4)}")

NameError: name 'squat_videos' is not defined

In [ ]:
for exercise_name, video_list in all_exercises:
    leans_all = []
    print(f"\n{exercise_name}:")
    for p in video_list:
        leans = compute_alignment_deviation(p, "trunk_lean")
        valid = [l for l in leans if l is not None]
        leans_all.extend(valid)

    arr = np.array(leans_all)
    print(f"  min:             {round(float(np.min(arr)), 4)}")
    print(f"  25th percentile: {round(float(np.percentile(arr, 25)), 4)}")
    print(f"  50th percentile: {round(float(np.percentile(arr, 50)), 4)}")
    print(f"  75th percentile: {round(float(np.percentile(arr, 75)), 4)}")
    print(f"  90th percentile: {round(float(np.percentile(arr, 90)), 4)}")
    print(f"  95th percentile: {round(float(np.percentile(arr, 95)), 4)}")
    print(f"  max:             {round(float(np.max(arr)), 4)}")

In [ ]:
def compute_back_angle(video_path):
    """
    Measures the angle at the hip between:
      shoulder → hip → knee
    
    ~180° = perfectly straight back
    < 160° = rounding forward
    > 200° = hyperextension (not possible with unsigned angle)
    
    Uses unsigned angle so:
      angle close to 180° = straight
      angle < threshold   = deviation detected
    
    Works from SIDE VIEW.
    
    Keypoints:
      shoulder midpoint = avg of (5, 6)
      hip midpoint      = avg of (11, 12)
      knee midpoint     = avg of (13, 14)
    """
    results = model.track(source=video_path, stream=True, verbose=False)
    angles = []

    for frame_result in results:
        if frame_result.keypoints is None:
            angles.append(None)
            continue
        if len(frame_result.keypoints.xy) == 0:
            angles.append(None)
            continue

        xy   = frame_result.keypoints.xy[0].cpu().numpy()
        conf = frame_result.keypoints.conf[0].cpu().numpy()

        needed = [5, 6, 11, 12, 13, 14]
        if any(conf[i] < CONF_THRESHOLD for i in needed):
            angles.append(None)
            continue

        # Midpoints
        shoulder = (xy[5] + xy[6]) / 2
        hip      = (xy[11] + xy[12]) / 2
        knee     = (xy[13] + xy[14]) / 2

        # Angle at hip between shoulder→hip→knee
        v1 = shoulder - hip
        v2 = knee     - hip

        cos_angle = np.dot(v1, v2) / (
            np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-6
        )
        angle = float(np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0))))
        angles.append(round(angle, 2))

    return angles

In [ ]:
for exercise_name, video_list in all_exercises:
    angles_all = []
    print(f"\n{exercise_name}:")
    for p in video_list:
        angles = compute_back_angle(p)
        valid  = [a for a in angles if a is not None]
        angles_all.extend(valid)

    arr = np.array(angles_all)
    print(f"  min:             {round(float(np.min(arr)), 4)}")
    print(f"  25th percentile: {round(float(np.percentile(arr, 25)), 4)}")
    print(f"  50th percentile: {round(float(np.percentile(arr, 50)), 4)}")
    print(f"  75th percentile: {round(float(np.percentile(arr, 75)), 4)}")
    print(f"  90th percentile: {round(float(np.percentile(arr, 90)), 4)}")
    print(f"  95th percentile: {round(float(np.percentile(arr, 95)), 4)}")
    print(f"  max:             {round(float(np.max(arr)), 4)}")

In [ ]:
squat_reference = {
    "knee_width_min":    1.5,
    "lean_back_min":    -0.3714,
    "lean_forward_max":  0.2678,
    "back_angle_min":   70.1575
}

deadlift_reference = {
    "knee_width_min":    1.45,
    "lean_back_min":    -0.1567,
    "lean_forward_max":  0.2785,
    "back_angle_min":   161.25
}

rdl_reference = {
    "knee_width_min":   1.6157,
    "lean_back_min":   -0.8061,
    "lean_forward_max": 0.6549,
    "back_angle_min":   76.6325
}

with open("squat_reference.json", "w") as f:
    json.dump(squat_reference, f, indent=2)

with open("deadlift_reference.json", "w") as f:
    json.dump(deadlift_reference, f, indent=2)

with open("romanian_deadlift_reference.json", "w") as f:
    json.dump(rdl_reference, f, indent=2)

print("squat_reference.json:")
print(json.dumps(squat_reference, indent=2))
print("\ndeadlift_reference.json:")
print(json.dumps(deadlift_reference, indent=2))
print("\nromanian_deadlift_reference.json:")
print(json.dumps(rdl_reference, indent=2))

In [ ]:
import cv2
import numpy as np
import json
from PIL import Image
import os

# ── Load reference ────────────────────────────────────────────
with open("squat_reference.json") as f:
    reference = json.load(f)

KNEE_WIDTH_MIN   = reference["knee_width_min"]   
LEAN_BACK_MIN    = reference["lean_back_min"]    - 0.05
LEAN_FORWARD_MAX = reference["lean_forward_max"] + 0.05
BACK_ANGLE_MIN   = reference["back_angle_min"]   - 5.0
CONF_THRESHOLD   = 0.3
DEBOUNCE_FRAMES  = 3

# ── Load GIF frames ───────────────────────────────────────────
gif_path = "/kaggle/input/datasets/abdallahgalalll/test-set/squat.gif"

gif = Image.open(gif_path)
frames = []
try:
    while True:
        frame_rgb = gif.convert("RGB")
        frame_np  = np.array(frame_rgb)
        frame_bgr = cv2.cvtColor(frame_np, cv2.COLOR_RGB2BGR)
        frames.append(frame_bgr)
        gif.seek(gif.tell() + 1)
except EOFError:
    pass

print(f"GIF has {len(frames)} frames")

# ── Debounce counters ─────────────────────────────────────────
knee_count        = 0
lean_forward_count = 0
lean_back_count   = 0
back_count        = 0

knee_active        = False
lean_forward_active = False
lean_back_active   = False
back_active        = False

output_frames = []

for i, frame in enumerate(frames):
    result = model(frame, verbose=False)

    knee_ratio   = None
    trunk_lean   = None
    back_angle   = None
    kp_xy        = None
    kp_conf      = None

    # keypoint draw points
    left_knee_pt  = None
    right_knee_pt = None
    shoulder_pt   = None
    hip_pt        = None
    knee_mid_pt   = None

    if result and result[0].keypoints is not None:
        if len(result[0].keypoints.xy) > 0:
            xy   = result[0].keypoints.xy[0].cpu().numpy()
            conf = result[0].keypoints.conf[0].cpu().numpy()
            kp_xy   = xy
            kp_conf = conf

            # ── Knee width ratio ──────────────────────────────
            if all(conf[i] >= CONF_THRESHOLD for i in [11, 12, 13, 14]):
                left_knee  = xy[13]
                right_knee = xy[14]
                left_hip   = xy[11]
                right_hip  = xy[12]

                hip_width  = np.linalg.norm(left_hip - right_hip) + 1e-6
                knee_width = np.linalg.norm(left_knee - right_knee)
                knee_ratio = round(float(knee_width / hip_width), 4)

                left_knee_pt  = (int(left_knee[0]),  int(left_knee[1]))
                right_knee_pt = (int(right_knee[0]), int(right_knee[1]))

            # ── Trunk lean ────────────────────────────────────
            use_left     = conf[7] >= conf[8]
            shoulder_idx = 5  if use_left else 6
            hip_idx      = 11 if use_left else 12

            if all(conf[i] >= CONF_THRESHOLD for i in [shoulder_idx, hip_idx]):
                shoulder = xy[shoulder_idx]
                hip      = xy[hip_idx]
                torso    = np.linalg.norm(shoulder - hip) + 1e-6
                trunk_lean = round(float((shoulder[0] - hip[0]) / torso), 4)

                shoulder_pt = (int(shoulder[0]), int(shoulder[1]))
                hip_pt      = (int(hip[0]),      int(hip[1]))

            # ── Back angle ────────────────────────────────────
            if all(conf[i] >= CONF_THRESHOLD for i in [5, 6, 11, 12, 13, 14]):
                shoulder_mid = (xy[5] + xy[6]) / 2
                hip_mid      = (xy[11] + xy[12]) / 2
                knee_mid     = (xy[13] + xy[14]) / 2

                v1 = shoulder_mid - hip_mid
                v2 = knee_mid     - hip_mid

                cos_a = np.dot(v1, v2) / (
                    np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-6
                )
                back_angle = float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))

                knee_mid_pt = (int(hip_mid[0]), int(hip_mid[1]))

    # ── Debounce — knee cave ──────────────────────────────────
    if knee_ratio is not None and knee_ratio < KNEE_WIDTH_MIN:
        knee_count += 1
    else:
        knee_count  = 0
        knee_active = False
    if knee_count >= DEBOUNCE_FRAMES:
        knee_active = True

    # ── Debounce — lean forward ───────────────────────────────
    if trunk_lean is not None and trunk_lean > LEAN_FORWARD_MAX:
        lean_forward_count += 1
    else:
        lean_forward_count  = 0
        lean_forward_active = False
    if lean_forward_count >= DEBOUNCE_FRAMES:
        lean_forward_active = True

    # ── Debounce — lean back ──────────────────────────────────
    if trunk_lean is not None and trunk_lean < LEAN_BACK_MIN:
        lean_back_count += 1
    else:
        lean_back_count  = 0
        lean_back_active = False
    if lean_back_count >= DEBOUNCE_FRAMES:
        lean_back_active = True

    # ── Debounce — back rounding ──────────────────────────────
    if back_angle is not None and back_angle < BACK_ANGLE_MIN:
        back_count  += 1
    else:
        back_count  = 0
        back_active = False
    if back_count >= DEBOUNCE_FRAMES:
        back_active = True

    # ── Draw on frame ─────────────────────────────────────────
    annotated = frame.copy()

    # Draw keypoints
    if kp_xy is not None:
        for idx in [5, 6, 7, 8, 11, 12, 13, 14]:
            if kp_conf[idx] >= CONF_THRESHOLD:
                pt = (int(kp_xy[idx][0]), int(kp_xy[idx][1]))
                cv2.circle(annotated, pt, 5, (0, 255, 0), -1)

    # Draw knee line
    if left_knee_pt and right_knee_pt:
        color = (0, 0, 255) if knee_active else (0, 255, 0)
        cv2.line(annotated, left_knee_pt, right_knee_pt, color, 2)

    # Draw trunk line
    if shoulder_pt and hip_pt:
        color = (0, 0, 255) if (lean_forward_active or lean_back_active) else (255, 165, 0)
        cv2.line(annotated, shoulder_pt, hip_pt, color, 2)

    # ── Values display ────────────────────────────────────────
    y = 25
    if knee_ratio is not None:
        cv2.putText(annotated, f"knee ratio: {knee_ratio:.3f} min={KNEE_WIDTH_MIN:.3f}",
            (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 2)
        y += 22
    if trunk_lean is not None:
        cv2.putText(annotated, f"lean: {trunk_lean:.3f} [{LEAN_BACK_MIN:.3f},{LEAN_FORWARD_MAX:.3f}]",
            (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 2)
        y += 22
    if back_angle is not None:
        cv2.putText(annotated, f"back angle: {back_angle:.1f} min={BACK_ANGLE_MIN:.1f}",
            (10, y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 2)
        y += 22

    # ── Feedback banners ──────────────────────────────────────
    banner_y      = y + 5
    any_violation = knee_active or lean_forward_active or lean_back_active or back_active

    if knee_active:
        cv2.rectangle(annotated, (0, banner_y), (annotated.shape[1], banner_y+35), (0,0,200), -1)
        cv2.putText(annotated, "WARNING: Knees caving in!",
            (10, banner_y+24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
        banner_y += 38

    if lean_forward_active:
        cv2.rectangle(annotated, (0, banner_y), (annotated.shape[1], banner_y+35), (0,100,200), -1)
        cv2.putText(annotated, "WARNING: Leaning too far forward!",
            (10, banner_y+24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
        banner_y += 38

    if lean_back_active:
        cv2.rectangle(annotated, (0, banner_y), (annotated.shape[1], banner_y+35), (0,100,200), -1)
        cv2.putText(annotated, "WARNING: Leaning too far back!",
            (10, banner_y+24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
        banner_y += 38

    if back_active:
        cv2.rectangle(annotated, (0, banner_y), (annotated.shape[1], banner_y+35), (0,0,200), -1)
        cv2.putText(annotated, "WARNING: Back rounding — keep spine straight!",
            (10, banner_y+24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
        banner_y += 38

    if not any_violation:
        cv2.rectangle(annotated, (0, banner_y), (annotated.shape[1], banner_y+35), (0,150,0), -1)
        cv2.putText(annotated, "Good form",
            (10, banner_y+24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

    output_frames.append(annotated)

# ── Save output ───────────────────────────────────────────────
h, w = output_frames[0].shape[:2]
out  = cv2.VideoWriter(
    "feedback_squat.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    10,
    (w, h)
)
for frame in output_frames:
    out.write(frame)
out.release()
print("Saved: feedback_squat.mp4")

In [ ]:
def compute_elbow_angle(video_path):
    results = model.track(source=video_path, stream=True, verbose=False)
    angles = []

    for frame_result in results:
        if frame_result.keypoints is None:
            angles.append(None)
            continue
        if len(frame_result.keypoints.xy) == 0:
            angles.append(None)
            continue

        xy   = frame_result.keypoints.xy[0].cpu().numpy()
        conf = frame_result.keypoints.conf[0].cpu().numpy()

        use_left     = conf[7] >= conf[8]
        shoulder_idx = 5  if use_left else 6
        elbow_idx    = 7  if use_left else 8
        wrist_idx    = 9  if use_left else 10

        needed = [shoulder_idx, elbow_idx, wrist_idx]
        if any(conf[i] < CONF_THRESHOLD for i in needed):
            angles.append(None)
            continue

        shoulder = xy[shoulder_idx]
        elbow    = xy[elbow_idx]
        wrist    = xy[wrist_idx]

        # ── Only measure at bottom of press ───────────────────
        # Bottom = elbow is BELOW shoulder in pixel coordinates
        # (y increases downward so elbow_y > shoulder_y = below)
        if elbow[1] <= shoulder[1]:
            # Elbow is above or at shoulder level = top of press
            # Skip this frame — not the bottom position
            angles.append(None)
            continue

        v1 = shoulder - elbow
        v2 = wrist    - elbow

        cos_a = np.dot(v1, v2) / (
            np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-6
        )
        angle = float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))
        angles.append(round(angle, 2))

    return angles

In [ ]:
def compute_elbow_horizontal_position(video_path):
    """
    Measures how far elbow is in front of or behind shoulder.
    
    offset = (elbow_x - shoulder_x) / torso_length
    
    positive = elbow in front of shoulder = safe ✅
    negative = elbow behind shoulder = danger ❌
    
    Automatically detects facing direction and flips sign.
    Works from SIDE VIEW.
    """
    results = model.track(source=video_path, stream=True, verbose=False)
    offsets = []

    for frame_result in results:
        if frame_result.keypoints is None:
            offsets.append(None)
            continue
        if len(frame_result.keypoints.xy) == 0:
            offsets.append(None)
            continue

        xy   = frame_result.keypoints.xy[0].cpu().numpy()
        conf = frame_result.keypoints.conf[0].cpu().numpy()

        use_left     = conf[7] >= conf[8]
        shoulder_idx = 5  if use_left else 6
        elbow_idx    = 7  if use_left else 8
        hip_idx      = 11 if use_left else 12

        needed = [shoulder_idx, elbow_idx, hip_idx]
        if any(conf[i] < CONF_THRESHOLD for i in needed):
            offsets.append(None)
            continue

        shoulder = xy[shoulder_idx]
        elbow    = xy[elbow_idx]
        hip      = xy[hip_idx]

        # ── Detect facing direction ───────────────────────────
        if conf[0] >= CONF_THRESHOLD:
            facing_right = xy[0][0] > hip[0]
        else:
            facing_right = shoulder[0] > hip[0]

        torso_length = np.linalg.norm(shoulder - hip) + 1e-6
        raw_offset   = (elbow[0] - shoulder[0]) / torso_length

        # Flip sign if facing left so positive always = in front
        offset = round(float(raw_offset if facing_right else -raw_offset), 4)
        offsets.append(offset)

    return offsets

In [ ]:
bench_videos    = get_videos("bench press")
incline_videos  = get_videos("incline bench press")
shoulder_videos = get_videos("shoulder press")

all_press_videos = [
    ("bench_press",         bench_videos),
    ("incline_bench_press", incline_videos),
    ("shoulder_press",      shoulder_videos),
]

# ── Elbow angle distribution ──────────────────────────────────
print("=== ELBOW ANGLE ===")
all_angles = []
for exercise_name, video_list in all_press_videos:
    print(f"\n{exercise_name}:")
    for p in video_list:
        angles = compute_elbow_angle(p)
        valid  = [a for a in angles if a is not None]
        print(f"  {os.path.basename(p)}: {len(valid)} valid frames")
        all_angles.extend(valid)

arr = np.array(all_angles)
print(f"\nTotal valid frames: {len(all_angles)}")
print(f"  min:             {round(float(np.min(arr)), 4)}")
print(f"  25th percentile: {round(float(np.percentile(arr, 25)), 4)}")
print(f"  50th percentile: {round(float(np.percentile(arr, 50)), 4)}")
print(f"  75th percentile: {round(float(np.percentile(arr, 75)), 4)}")
print(f"  90th percentile: {round(float(np.percentile(arr, 90)), 4)}")
print(f"  95th percentile: {round(float(np.percentile(arr, 95)), 4)}")
print(f"  max:             {round(float(np.max(arr)), 4)}")

# ── Elbow horizontal offset distribution ─────────────────────
print("\n=== ELBOW HORIZONTAL OFFSET ===")
all_offsets = []
for exercise_name, video_list in all_press_videos:
    print(f"\n{exercise_name}:")
    for p in video_list:
        offsets = compute_elbow_horizontal_position(p)
        valid   = [o for o in offsets if o is not None]
        print(f"  {os.path.basename(p)}: {len(valid)} valid frames")
        all_offsets.extend(valid)

arr = np.array(all_offsets)
print(f"\nTotal valid frames: {len(all_offsets)}")
print(f"  min:             {round(float(np.min(arr)), 4)}")
print(f"  25th percentile: {round(float(np.percentile(arr, 25)), 4)}")
print(f"  50th percentile: {round(float(np.percentile(arr, 50)), 4)}")
print(f"  75th percentile: {round(float(np.percentile(arr, 75)), 4)}")
print(f"  90th percentile: {round(float(np.percentile(arr, 90)), 4)}")
print(f"  95th percentile: {round(float(np.percentile(arr, 95)), 4)}")
print(f"  max:             {round(float(np.max(arr)), 4)}")

In [ ]:
press_reference = {
    "elbow_offset_min": 0.05
}

with open("press_reference.json", "w") as f:
    json.dump(press_reference, f, indent=2)

print(json.dumps(press_reference, indent=2))

In [19]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("pratapdevs11/gym-data-coco-format-for-yolo-pose")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/pratapdevs11/gym-data-coco-format-for-yolo-pose


In [22]:
import json
import csv
from collections import defaultdict


IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
DATASET_ROOT = os.path.join(path,'JIM_DATA29')
OUTPUT_DIR   = "/kaggle/working/references"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EXERCISES = {
    "deadlift":          os.path.join(DATASET_ROOT, "Deadlifts/deadlift Data29"),
    "romanian_deadlift": os.path.join(DATASET_ROOT, "Deadlifts/romania Data29"),
    "sumo_deadlift":     os.path.join(DATASET_ROOT, "Deadlifts/sumo data 29"),
    "squat":             os.path.join(DATASET_ROOT, "Squats/squats data29"),
    "front_squat":       os.path.join(DATASET_ROOT, "Squats/front_squat data29"),
    "zercher_squat":     os.path.join(DATASET_ROOT, "Squats/zercher Data29"),
}

print("Exercise folders:")
for name, p in EXERCISES.items():
    exists = os.path.exists(p)
    if exists:
        # count images in train subfolder if exists
        train_path = os.path.join(p, "images", "train")
        if os.path.exists(train_path):
            count = sum(1 for f in os.listdir(train_path)
                       if os.path.splitext(f)[1].lower() in IMG_EXTS)
            print(f"  {name}: {count} train images")
        else:
            print(f"  {name}: exists (no images/train subfolder)")
    else:
        print(f"  {name}: NOT FOUND")

Exercise folders:
  deadlift: exists (no images/train subfolder)
  romanian_deadlift: 315 train images
  sumo_deadlift: 510 train images
  squat: 492 train images
  front_squat: 630 train images
  zercher_squat: 411 train images


In [25]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Image iterator (train only)
# ═══════════════════════════════════════════════════════════════
def iter_images(exercise_path):
    """
    Yields image paths from exercise folder.
    Prefers images/train subfolder if it exists.
    Falls back to walking entire directory.
    """
    train_path = os.path.join(exercise_path, "images", "train")
    if os.path.exists(train_path):
        for fn in os.listdir(train_path):
            if os.path.splitext(fn)[1].lower() in IMG_EXTS:
                yield os.path.join(train_path, fn)
    else:
        for dirpath, _, filenames in os.walk(exercise_path):
            # Skip label folders
            if "labels" in dirpath:
                continue
            for fn in filenames:
                if os.path.splitext(fn)[1].lower() in IMG_EXTS:
                    yield os.path.join(dirpath, fn)

In [26]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — Keypoint extraction
# ═══════════════════════════════════════════════════════════════
def extract_keypoints(exercise_name, exercise_path):
    """
    Runs YOLO on all train images for an exercise.
    Returns list of dicts with xy and conf per keypoint.
    Only includes frames where person is detected.
    """
    frames = []
    images = list(iter_images(exercise_path))
    print(f"\n{exercise_name}: {len(images)} images")

    for i, img_path in enumerate(images):
        if i % 100 == 0:
            print(f"  {i}/{len(images)}...")
        try:
            result = model(img_path, verbose=False)
            if not result or result[0].keypoints is None:
                continue
            if len(result[0].keypoints.xy) == 0:
                continue

            xy   = result[0].keypoints.xy[0].cpu().numpy()
            conf = result[0].keypoints.conf[0].cpu().numpy()

            frames.append({
                "xy":   xy,
                "conf": conf,
                "path": img_path
            })
        except Exception as e:
            print(f"  ERROR {img_path}: {e}")

    print(f"  → {len(frames)} valid frames")
    return frames

In [29]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — Angle and metric computation
# ═══════════════════════════════════════════════════════════════
def calculate_angle(a, b, c):
    """Angle in degrees at vertex b."""
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b
    cos_a = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))



def compute_metrics(frames):
    """
    Computes per-frame metrics from keypoint frames.

    Metrics:
      hip_angle_L/R
      knee_angle_L/R
      asymmetry
      knee_ankle_ratio
      trunk_lean

    Added:
      - knee_ratio (same as knee_ankle_ratio but used for phase analysis)
      - phase detection (top / mid / bottom using knee angle)
      - knee_ratio per phase
      - knee_collapse_ratio (bottom vs top)
    """
    all_values = defaultdict(list)

    # Temporary storage for phase detection
    temp_knee_angles = []
    temp_knee_ratios = []

    for frame in frames:
        xy   = frame["xy"]
        conf = frame["conf"]

        def kp(idx, threshold=CONF_MIN):
            if conf[idx] >= threshold:
                return xy[idx]
            return None

        # ── Keypoints ─────────────────────────────
        shL  = kp(5);  shR  = kp(6)
        hipL = kp(11); hipR = kp(12)
        kneL = kp(13); kneR = kp(14)
        ankL = kp(15); ankR = kp(16)

        # ── Hip angles ────────────────────────────
        if shL is not None and hipL is not None and kneL is not None:
            all_values["hip_angle_L"].append(
                calculate_angle(shL, hipL, kneL)
            )
        if shR is not None and hipR is not None and kneR is not None:
            all_values["hip_angle_R"].append(
                calculate_angle(shR, hipR, kneR)
            )

        # ── Knee angles ───────────────────────────
        kaL = None
        kaR = None

        if hipL is not None and kneL is not None and ankL is not None:
            kaL = calculate_angle(hipL, kneL, ankL)
            all_values["knee_angle_L"].append(kaL)

        if hipR is not None and kneR is not None and ankR is not None:
            kaR = calculate_angle(hipR, kneR, ankR)
            all_values["knee_angle_R"].append(kaR)

        # Average knee angle (for phase detection)
        if kaL is not None and kaR is not None:
            knee_angle_avg = (kaL + kaR) / 2.0
        elif kaL is not None:
            knee_angle_avg = kaL
        elif kaR is not None:
            knee_angle_avg = kaR
        else:
            knee_angle_avg = None

        # ── Asymmetry ─────────────────────────────
        if kaL is not None and kaR is not None:
            all_values["asymmetry"].append(abs(kaL - kaR))

        # ── Knee ankle ratio (original) ───────────
        kneL_h = kp(13, 0.5); kneR_h = kp(14, 0.5)
        ankL_h = kp(15, 0.5); ankR_h = kp(16, 0.5)

        knee_ratio = None
        if all(v is not None for v in [kneL_h, kneR_h, ankL_h, ankR_h]):
            ankle_width = np.linalg.norm(ankL_h - ankR_h) + 1e-6
            knee_width  = np.linalg.norm(kneL_h - kneR_h)

            knee_ratio = float(knee_width / ankle_width)
            all_values["knee_ankle_ratio"].append(round(knee_ratio, 4))

        # ── Trunk lean (KEPT as requested) ────────
        use_left     = conf[11] >= conf[12]
        shoulder_idx = 5  if use_left else 6
        hip_idx      = 11 if use_left else 12

        sh_h  = kp(shoulder_idx, 0.5)
        hip_h = kp(hip_idx, 0.5)

        if sh_h is not None and hip_h is not None:
            torso = np.linalg.norm(sh_h - hip_h) + 1e-6

            # Facing direction logic (still kept)
            nose = kp(0, 0.5)
            if nose is not None:
                facing_right = bool(nose[0] > hip_h[0])
            else:
                facing_right = bool(sh_h[0] > hip_h[0])

            raw_lean = (sh_h[0] - hip_h[0]) / torso
            trunk_lean = float(raw_lean if facing_right else -raw_lean)

            all_values["trunk_lean"].append(round(trunk_lean, 4))

        # Store for phase processing
        if knee_angle_avg is not None:
            temp_knee_angles.append(knee_angle_avg)
            temp_knee_ratios.append(knee_ratio)

    # ─────────────────────────────────────────────
    # Phase Detection (approximation)
    # ─────────────────────────────────────────────
    if len(temp_knee_angles) > 0:
        angles = np.array(temp_knee_angles)

        low_thr  = np.percentile(angles, 20)  # bottom
        high_thr = np.percentile(angles, 80)  # top

        top_ratios = []
        mid_ratios = []
        bottom_ratios = []

        for angle, ratio in zip(temp_knee_angles, temp_knee_ratios):
            if ratio is None:
                continue

            if angle <= low_thr:
                bottom_ratios.append(ratio)
            elif angle >= high_thr:
                top_ratios.append(ratio)
            else:
                mid_ratios.append(ratio)

        # Store phase data
        if top_ratios:
            all_values["knee_ratio_top"] = top_ratios
        if mid_ratios:
            all_values["knee_ratio_mid"] = mid_ratios
        if bottom_ratios:
            all_values["knee_ratio_bottom"] = bottom_ratios

        # ── Collapse ratio ────────────────────────
        if top_ratios and bottom_ratios:
            top_mean = np.mean(top_ratios)
            bottom_mean = np.mean(bottom_ratios)

            collapse_ratio = float(bottom_mean / (top_mean + 1e-6))
            all_values["knee_collapse_ratio"].append(round(collapse_ratio, 4))

    return all_values
  

def summarize(values):
    """Compute percentile summary for a list of values."""
    arr = np.array(values)
    return {
        "count": int(arr.size),
        "min":   round(float(np.min(arr)), 4),
        "25th":  round(float(np.percentile(arr, 25)), 4),
        "50th":  round(float(np.percentile(arr, 50)), 4),
        "75th":  round(float(np.percentile(arr, 75)), 4),
        "90th":  round(float(np.percentile(arr, 90)), 4),
        "95th":  round(float(np.percentile(arr, 95)), 4),
        "max":   round(float(np.max(arr)), 4),
    }

In [31]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — Generate reference JSON per exercise
# ═══════════════════════════════════════════════════════════════
def generate_reference(all_values, exercise_name):
    """
    Generates reference thresholds from metric distributions.

    Thresholds represent the FLOOR of safe professional movement:
      hip_angle_min             = 25th percentile → below = dangerous lean
      knee_angle_min            = 25th percentile → below = dangerous depth
      asymmetry_max             = 90th percentile → above = dangerous asymmetry
      knee_ankle_ratio_min      = 25th percentile → below = knees caving
      trunk_lean_max            = 90th percentile → above = excessive lean

    NEW:
      knee_collapse_ratio_min   = 25th percentile → below = knees collapsing inward
    """
    ref = {"exercise": exercise_name}

    # ── Hip angle (forward lean) ──────────────────────────────
    hip_vals = []
    for side in ["hip_angle_L", "hip_angle_R"]:
        if all_values.get(side):
            hip_vals.extend(all_values[side])
    if hip_vals:
        arr = np.array(hip_vals)
        ref["hip_angle_min"] = round(float(np.percentile(arr, 25)), 2)
        ref["hip_angle_summary"] = summarize(hip_vals)

    # ── Knee angle (depth) ────────────────────────────────────
    knee_vals = []
    for side in ["knee_angle_L", "knee_angle_R"]:
        if all_values.get(side):
            knee_vals.extend(all_values[side])
    if knee_vals:
        arr = np.array(knee_vals)
        ref["knee_angle_min"] = round(float(np.percentile(arr, 25)), 2)
        ref["knee_angle_summary"] = summarize(knee_vals)

    # ── Asymmetry ─────────────────────────────────────────────
    if all_values.get("asymmetry"):
        arr = np.array(all_values["asymmetry"])
        ref["asymmetry_max"] = round(float(np.percentile(arr, 90)), 2)
        ref["asymmetry_summary"] = summarize(all_values["asymmetry"])

    # ── Knee ankle ratio (static check) ───────────────────────
    if all_values.get("knee_ankle_ratio"):
        arr = np.array(all_values["knee_ankle_ratio"])
        ref["knee_ankle_ratio_min"] = round(float(np.percentile(arr, 25)), 4)
        ref["knee_ankle_ratio_summary"] = summarize(all_values["knee_ankle_ratio"])

    # ── Trunk lean ────────────────────────────────────────────
    if all_values.get("trunk_lean"):
        arr = np.array(all_values["trunk_lean"])
        ref["trunk_lean_max"] = round(float(np.percentile(arr, 90)), 4)
        ref["trunk_lean_summary"] = summarize(all_values["trunk_lean"])

    # ─────────────────────────────────────────────────────────
    # NEW — Phase-based knee collapse detection
    # ─────────────────────────────────────────────────────────

    # Collapse ratio (main signal)
    if all_values.get("knee_collapse_ratio"):
        arr = np.array(all_values["knee_collapse_ratio"])
        ref["knee_collapse_ratio_min"] = round(float(np.percentile(arr, 25)), 4)
        ref["knee_collapse_ratio_summary"] = summarize(all_values["knee_collapse_ratio"])

    # Optional: store phase-specific distributions (useful for debugging/analysis)
    if all_values.get("knee_ratio_top"):
        ref["knee_ratio_top_summary"] = summarize(all_values["knee_ratio_top"])

    if all_values.get("knee_ratio_mid"):
        ref["knee_ratio_mid_summary"] = summarize(all_values["knee_ratio_mid"])

    if all_values.get("knee_ratio_bottom"):
        ref["knee_ratio_bottom_summary"] = summarize(all_values["knee_ratio_bottom"])

    return ref


# ── Run pipeline for all exercises ───────────────────────────
all_references = {}
CONF_MIN=CONF_THRESHOLD
for exercise_name, exercise_path in EXERCISES.items():
    if not os.path.exists(exercise_path):
        print(f"\nSkipping {exercise_name} — path not found")
        continue

    # Step 1: Extract keypoints
    frames = extract_keypoints(exercise_name, exercise_path)

    if not frames:
        print(f"  No valid frames for {exercise_name}")
        continue

    # Step 2: Compute metrics
    all_values = compute_metrics(frames)

    # Step 3: Generate reference
    ref = generate_reference(all_values, exercise_name)
    all_references[exercise_name] = ref

    # Step 4: Save reference JSON
    out_path = os.path.join(OUTPUT_DIR, f"{exercise_name}_reference.json")
    with open(out_path, "w") as f:
        json.dump(ref, f, indent=2)

    # Print summary
    print(f"\n{'='*50}")
    print(f"Reference: {exercise_name}")
    print(f"{'='*50}")
    for key, val in ref.items():
        if not key.endswith("_summary") and key != "exercise":
            print(f"  {key}: {val}")

print("\n\nAll references saved to:", OUTPUT_DIR)


deadlift: 818 images
  0/818...
  100/818...
  200/818...
  300/818...
  400/818...
  500/818...
  600/818...
  700/818...
  800/818...
  → 818 valid frames

Reference: deadlift
  hip_angle_min: 162.79
  knee_angle_min: 166.84
  asymmetry_max: 18.5
  knee_ankle_ratio_min: 1.276
  trunk_lean_max: -0.1241
  knee_collapse_ratio_min: 1.1416

romanian_deadlift: 315 images
  0/315...
  100/315...
  200/315...
  300/315...
  → 315 valid frames

Reference: romanian_deadlift
  hip_angle_min: 166.53
  knee_angle_min: 172.76
  asymmetry_max: 6.39
  knee_ankle_ratio_min: 1.1507
  trunk_lean_max: -0.1215
  knee_collapse_ratio_min: 1.242

sumo_deadlift: 510 images
  0/510...
  100/510...
  200/510...
  300/510...
  400/510...
  500/510...
  → 510 valid frames

Reference: sumo_deadlift
  hip_angle_min: 109.76
  knee_angle_min: 131.47
  asymmetry_max: 50.6
  knee_ankle_ratio_min: 0.7137
  trunk_lean_max: 0.0216
  knee_collapse_ratio_min: 1.1116

squat: 492 images
  0/492...
  100/492...
  200/492...


In [32]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — Print full distributions for review (ENHANCED)
# ═══════════════════════════════════════════════════════════════
for exercise_name, ref in all_references.items():
    print(f"\n{'='*50}")
    print(f"Full distributions: {exercise_name}")
    print(f"{'='*50}")

    # Highlight important new metric first
    if "knee_collapse_ratio_summary" in ref:
        print("\n🔥 Knee Collapse (MOST IMPORTANT):")
        for k, v in ref["knee_collapse_ratio_summary"].items():
            print(f"    {k}: {v}")

    # Print phase-based metrics
    for phase in ["top", "mid", "bottom"]:
        key = f"knee_ratio_{phase}_summary"
        if key in ref:
            print(f"\n📊 Knee Ratio ({phase.upper()} phase):")
            for k, v in ref[key].items():
                print(f"    {k}: {v}")

    # Print remaining summaries
    for key, val in ref.items():
        if key.endswith("_summary") and not any(x in key for x in [
            "knee_collapse_ratio",
            "knee_ratio_top",
            "knee_ratio_mid",
            "knee_ratio_bottom"
        ]):
            metric_name = key.replace("_summary", "")
            print(f"\n  {metric_name}:")
            for k, v in val.items():
                print(f"    {k}: {v}")


Full distributions: deadlift

🔥 Knee Collapse (MOST IMPORTANT):
    count: 1
    min: 1.1416
    25th: 1.1416
    50th: 1.1416
    75th: 1.1416
    90th: 1.1416
    95th: 1.1416
    max: 1.1416

📊 Knee Ratio (TOP phase):
    count: 164
    min: 1.1605
    25th: 1.1896
    50th: 1.2045
    75th: 1.2245
    90th: 1.2463
    95th: 1.2649
    max: 1.2745

📊 Knee Ratio (MID phase):
    count: 490
    min: 1.2126
    25th: 1.3256
    50th: 1.3743
    75th: 1.4278
    90th: 1.4565
    95th: 1.4818
    max: 1.5999

📊 Knee Ratio (BOTTOM phase):
    count: 164
    min: 1.2376
    25th: 1.3212
    50th: 1.3764
    75th: 1.4158
    90th: 1.4453
    95th: 1.5055
    max: 1.6056

  hip_angle:
    count: 1636
    min: 122.1251
    25th: 162.7887
    50th: 168.5505
    75th: 171.4986
    90th: 174.403
    95th: 174.9176
    max: 177.5636

  knee_angle:
    count: 1636
    min: 141.4545
    25th: 166.8358
    50th: 170.8138
    75th: 173.4
    90th: 177.1452
    95th: 177.6757
    max: 179.9209

  asy